In [ ]:
import numpy as np
import pandas as pd

In [ ]:
def rbf_kernel(xi, xj, sigma=1):
    distance_squared = np.linalg.norm(xi - xj)**2
    return np.exp(-distance_squared / (2 * sigma**2))


In [ ]:
def compute_kernel_matrix(X, kernel):
    n = X.shape[0]
    K = np.zeros((n,n))

    for i in range(n):
        for j in range(n):
            K[i,j] = kernel(X[i], X[j])

    return K


In [ ]:
def kernel_svm(dataset, unseen_data,kernel):
    dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)

    split=int(0.7*len(dataset))

    train_data=dataset.iloc[:split]
    test_data=dataset.iloc[split:]

    x_train=train_data.iloc[:,:-1]
    y_train=train_data.iloc[:,-1].values

    x_test=test_data.iloc[:,:-1]
    y_test=test_data.iloc[:,-1].values

    mean=x_train.mean()
    std=x_train.std()

    x_train=(x_train-mean)/std
    x_test=(x_test-mean)/std

    x_train=x_train.values
    x_test=x_test.values

    y_train = np.where(y_train==0,-1,1)
    y_test = np.where(y_test==0,-1,1)

    n_samples = x_train.shape[0]
    alphas = np.zeros(n_samples)

    bias=np.random.rand()

    learning_rate=0.01
    lamda=0.01
    epochs=500
    K = compute_kernel_matrix(x_train, kernel)

    for epoch in range(1,epochs+1):
        indices = np.random.permutation(len(x_train))

        c=1/lamda

        for i in indices:
            prediction_i=np.sum(alphas * y_train * K[:,i])
            gradient = 1 - y_train[i] * prediction_i
            alphas[i] += learning_rate * gradient
            alphas[i]=np.clip(alphas[i],0,c)
        
    support_vectors = np.where(alphas > 1e-5)[0]

    bias_values = []

    for s in support_vectors:
        value = y_train[s] - np.sum(alphas * y_train * K[:, s])
        bias_values.append(value)

    bias = np.mean(bias_values)

    predictions = []

    for x in x_test:
        
        value = 0
        
        for i in range(n_samples):
            if alphas[i] > 1e-5:
                value += alphas[i] * y_train[i] * kernel(x_train[i], x)
        
        value += bias
        
        predictions.append(np.sign(value))

    predictions = np.array(predictions)

    accuracy = np.mean(predictions == y_test)
    print("Accuracy:", accuracy)

    unseen_predictions=[]

    unseen_data = unseen_data.values

    for x in unseen_data:
        
        value = 0
        
        for i in range(n_samples):
            if alphas[i] > 1e-5:
                value += alphas[i] * y_train[i] * kernel(x_train[i], x)
        
        value += bias
        
        unseen_predictions.append(np.sign(value))

    unseen_predictions = np.array(unseen_predictions)

    return unseen_predictions


In [ ]:
def generate_kernel_svm_dataset(n_samples=2000, noise=0.1, random_state=42):
    
    np.random.seed(random_state)
    
    n_outer = n_samples // 2
    n_inner = n_samples // 2
    
    # outer circle
    theta_outer = np.random.uniform(0, 2*np.pi, n_outer)
    r_outer = 2 + noise*np.random.randn(n_outer)
    
    x_outer = r_outer * np.cos(theta_outer)
    y_outer = r_outer * np.sin(theta_outer)
    
    # inner circle
    theta_inner = np.random.uniform(0, 2*np.pi, n_inner)
    r_inner = 1 + noise*np.random.randn(n_inner)
    
    x_inner = r_inner * np.cos(theta_inner)
    y_inner = r_inner * np.sin(theta_inner)
    
    X = np.vstack([
        np.column_stack((x_inner, y_inner)),
        np.column_stack((x_outer, y_outer))
    ])
    
    y = np.array([0]*n_inner + [1]*n_outer)
    
    dataset = pd.DataFrame(X, columns=["feature1","feature2"])
    dataset["label"] = y
    
    return dataset


In [ ]:
dataset = generate_kernel_svm_dataset(n_samples=2000)
unseen_data = generate_kernel_svm_dataset(n_samples=5)
unseen_data = unseen_data.iloc[:,:-1]


In [ ]:
import matplotlib.pyplot as plt

plt.scatter(dataset["feature1"], dataset["feature2"], c=dataset["label"])
plt.show()


In [ ]:
predictions=kernel_svm(dataset,unseen_data,rbf_kernel)
predictions

In [ ]:
unseen_data